In [12]:
# Install ReportLab and DejaVuSans fonts for Cyrillic support, and NotoColorEmoji for emojis
!pip install reportlab
!sudo apt-get update
!sudo apt-get install -y fonts-dejavu-core fonts-noto-color-emoji

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Fetched 3,917 B in 2s (1,617 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-dejavu-core is already the newest version (2

In [40]:
%%writefile sample_dna.txt
ATGCGTGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGCAGGCGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGCTGACGTACGTACGTACGTAGCATGCATGDNA

Writing sample_dna.txt


In [45]:
import xml.etree.ElementTree as ET
from urllib.error import URLError, HTTPError
from tqdm.auto import tqdm

from Bio import Entrez
import re
import os
import matplotlib.pyplot as plt

from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from google.colab import files # Needed for files.download

# --- НАСТРОЙКИ NCBI ---
# Используем нейтральный технический адрес проекта
Entrez.email = "genomic.atlas.project@example.com"

# --- БЛОК 1: ВИЗУАЛИЗАЦИЯ (ГЕНЕТИЧЕСКИЙ РАДАР) ---

def generate_gc_plot(sequence, window_size=20, output_img="gc_plot.png"):
    """Создает график распределения GC-состава (скользящего окна)."""
    values = []
    for i in range(len(sequence) - window_size + 1):
        subseq = sequence[i:i + window_size]
        gc = (subseq.count('G') + subseq.count('C')) / len(subseq) * 100
        values.append(gc)

    plt.figure(figsize=(8, 3))
    plt.plot(values, color='#2c3e50', linewidth=1.5, label='GC %')
    plt.fill_between(range(len(values)), values, color='#3498db', alpha=0.3)
    plt.axhline(y=50, color='r', linestyle='--', alpha=0.3) # Линия 50% (баланс)
    plt.title("Геномный Ландшафт: Радар GC-Состава", fontsize=12)
    plt.xlabel("Позиция в Последовательности (bp)", fontsize=10)
    plt.ylabel("GC %")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.savefig(output_img, dpi=100)
    plt.close()
    return output_img

# --- БЛОК 2: CLINVAR API ---

def get_clinvar_significance_for_list(rs_ids_list):
    """Пакетная проверка списка rs-номеров через NCBI ClinVar."""
    results = {}
    clinvar_ids = []
    id_map = {}

    if not rs_ids_list:
        return results

    print(f"🔍 Запрос данных для {len(rs_ids_list)} вариантов...")
    for rs_id in tqdm(rs_ids_list, desc="Поиск в базе"):
        try:
            handle = Entrez.esearch(db="clinvar", term=f"{rs_id}[Variant ID]")
            search_res = Entrez.read(handle)
            handle.close()
            if search_res['IdList']:
                cid = search_res['IdList'][0]
                clinvar_ids.append(cid)
                id_map[cid] = rs_id
            else:
                results[rs_id] = "Not Found"
        except Exception as e:
            print(f"❌ Ошибка при поиске '{rs_id}' в ClinVar: {e}")
            results[rs_id] = f"Error: {e}"

    if clinvar_ids:
        try:
            handle = Entrez.esummary(db="clinvar", id=",".join(clinvar_ids), validate=False)
            summaries = Entrez.read(handle, validate=False)
            handle.close()
            for doc in summaries['DocumentSummarySet']['DocumentSummary']:
                uid = str(doc.get('uid') or doc.attributes.get('uid'))
                orig_rs = id_map.get(uid)
                if orig_rs:
                    results[orig_rs] = doc['germline_classification']['description']
        except Exception as e:
            print(f"❌ Ошибка при получении сводки ClinVar: {e}")
            for cid in clinvar_ids:
                orig_rs = id_map.get(cid)
                if orig_rs and orig_rs not in results:
                    results[orig_rs] = f"Summary Error: {e}"
    return results





# --- БЛОК 3: ГЕНЕТИЧЕСКИЙ СКАНЕР ---

def universal_genomic_scanner(sequence):
    """Анализ сырой ДНК: GC-состав, повторы и мутации."""
    print("\n" + "="*45)
    print("🚀 STARTING GENOMIC SCAN v4.0 (Visual)")
    print("="*45)

    gc = (sequence.count('G') + sequence.count('C')) / len(sequence) * 100
    print(f"📊 Global GC-Content: {gc:.2f}%")

    htt_repeats_count = 0
    htt_repeats_status = "N/A"
    # Поиск CAG-повторов (Болезнь Гентингтона)
    repeats = re.findall(r'(?:CAG){3,}', sequence)
    if repeats:
        htt_repeats_count = len(max(repeats, key=len)) // 3
        htt_repeats_status = "PATHOGENIC" if htt_repeats_count >= 36 else "NORMAL"
        print(f"🧬 HTT Gene: {htt_repeats_count} CAG repeats ({htt_repeats_status})")

    sirt1_detected = False
    # Маркер долголетия SIRT1
    if sequence.startswith('G'):
        sirt1_detected = True
        print("🌟 SIRT1: Longevity Variant 'G' detected!")

    print("="*45 + "\n")
    return {
        "gc_content": gc,
        "htt_repeats_count": htt_repeats_count,
        "htt_repeats_status": htt_repeats_status,
        "sirt1_detected": sirt1_detected
    }

# Helper function to wrap text (re-added)
def wrap_text(text, font_name, font_size, max_width, canvas_obj):
    paragraphs = text.split('\n') # Split by explicit newline characters first
    all_wrapped_lines = []

    for paragraph in paragraphs:
        if not paragraph.strip(): # Handle empty lines (e.g., from multiple \n) as blank lines
            all_wrapped_lines.append('')
            continue

        words = paragraph.split(' ')
        current_line = []
        for word in words:
            test_line = ' '.join(current_line + [word])
            # Check if adding the next word exceeds max_width
            if canvas_obj.stringWidth(test_line, font_name, font_size) < max_width:
                current_line.append(word)
            else:
                # If current_line is not empty, add it to lines
                if current_line:
                    all_wrapped_lines.append(' '.join(current_line))
                # Start a new line with the current word.
                # If the current word itself is longer than max_width, it will be on its own line (and overflow if too long).
                current_line = [word]
        # Add any remaining words in current_line after the loop
        if current_line:
            all_wrapped_lines.append(' '.join(current_line))
    return all_wrapped_lines

# --- БЛОК 4: ГЕНЕРАЦИЯ PDF ---

def save_report_to_pdf(results, dna_sequence, scanner_data, exon_comparison_results, filename="Genomic_Report.pdf"):
    """Создание PDF отчета с графиком и таблицей."""
    # Register DejaVuSans for Cyrillic
    try:
        pdfmetrics.registerFont(TTFont('DejaVuSans', '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'))
        pdfmetrics.registerFont(TTFont('DejaVuSans-Bold', '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf'))
        font_name = 'DejaVuSans'
    except Exception:
        font_name = 'Helvetica'

    # Delete existing file if it exists
    if os.path.exists(filename):
        os.remove(filename)
        print(f"ℹ️ Existing file '{filename}' removed.")

    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter

    # Define thresholds for exon comparison interpretation
    NORMAL_SIMILARITY_THRESHOLD = 95
    POTENTIAL_RISK_SIMILARITY_THRESHOLD = 80

    # Заголовок отчета
    c.setFont(f"{font_name}-Bold" if font_name != 'Helvetica' else 'Helvetica-Bold', 18)
    c.drawCentredString(width / 2.0, height - 50, "Геномный Атлас Аномалий: Диагностический Отчет")

    # Визуализация (График вставляется под заголовком)
    c.setFont(font_name, 12)
    c.drawString(50, height - 85, "I. Анализ Ландшафта Последовательности:")
    img_path = generate_gc_plot(dna_sequence)
    c.drawImage(img_path, 50, height - 260, width=500, height=160)

    # Genomic Scan Summary Title
    summary_title_y = height - 270 # Below the image, with some padding
    c.setFont(f"{font_name}-Bold" if font_name != 'Helvetica' else 'Helvetica-Bold', 12)
    c.drawString(50, summary_title_y, "Сводка Геномного Сканирования:")

    current_y = summary_title_y - 20 # Below the summary title
    c.setFont(font_name, 10)

    # GC-Content line
    gc_content_line_en = f"Global GC-Content: {scanner_data['gc_content']:.2f}%"
    gc_content_line_ru = f"Генетический скан показал глобальное содержание GC {scanner_data['gc_content']:.2f}%"
    c.drawString(50, current_y, gc_content_line_en)
    current_y -= 12
    c.drawString(50, current_y, gc_content_line_ru)
    current_y -= 25 # Extra space after Russian translation

    # HTT Gene line
    if scanner_data['htt_repeats_count'] > 0:
        htt_status_display_en = ""
        htt_status_display_ru = ""
        if scanner_data['htt_repeats_status'] == "PATHOGENIC":
            htt_status_display_en = "(PATHOGENIC)"
            htt_status_display_ru = "(что расценивается как патогенный вариант)"
        elif scanner_data['htt_repeats_status'] == "NORMAL":
            htt_status_display_en = "(NORMAL)"
            htt_status_display_ru = "(что расценивается как нормальный вариант)"

        htt_line_en = f"HTT Gene: {scanner_data['htt_repeats_count']} CAG repeats {htt_status_display_en}"
        htt_line_ru = f"Обнаружено {scanner_data['htt_repeats_count']} повтора CAG в гене HTT {htt_status_display_ru}"

        c.drawString(50, current_y, htt_line_en)
        current_y -= 12
        c.drawString(50, current_y, htt_line_ru)
        current_y -= 25 # Extra space after Russian translation

    # SIRT1 line - Always display status
    sirt1_line_en = ""
    sirt1_line_ru = ""
    if scanner_data['sirt1_detected']:
        sirt1_line_en = "SIRT1: Longevity Variant 'G' detected!"
        sirt1_line_ru = "SIRT1: Обнаружен вариант 'G' долголетия."
    else:
        sirt1_line_en = "SIRT1: Longevity Variant 'G' NOT detected."
        sirt1_line_ru = "SIRT1: Вариант 'G' долголетия НЕ обнаружен."

    c.drawString(50, current_y, sirt1_line_en)
    current_y -= 12
    c.drawString(50, current_y, sirt1_line_ru)
    current_y -= 25 # Extra space after Russian translation

    # Сравнение с эталонными изоформами
    if exon_comparison_results:
        y = current_y - 20 # Position below the previous text
        c.setFont(f"{font_name}-Bold" if font_name != 'Helvetica' else 'Helvetica-Bold', 12)
        c.drawString(50, y, "III. Сравнение с эталонными изоформами:")
        y -= 20
        c.setFont(font_name, 10)

        for exon_name, result_data in exon_comparison_results.items():
            similarity = result_data['similarity']
            status = ""
            if similarity >= NORMAL_SIMILARITY_THRESHOLD:
                status = "Норма"
            elif similarity >= POTENTIAL_RISK_SIMILARITY_THRESHOLD:
                status = "Вариант обнаружен (Потенциальный риск)"
            else:
                status = "Патология (Высокий риск)"

            c.drawString(50, y, f"Сходство с {exon_name}: {similarity}% ({status})")
            y -= 15
        current_y = y - 10 # Update current_y for next section

    # Таблица клинической значимости
    y_start = current_y - 20 # Position below the summary text, with extra padding
    c.setFont(f"{font_name}-Bold" if font_name != 'Helvetica' else 'Helvetica-Bold', 12)
    c.drawString(50, y_start, "IV. Клиническая Значимость (ClinVar):")

    y = y_start - 30
    c.setFont(f"{font_name}-Bold" if font_name != 'Helvetica' else 'Helvetica-Bold', 10)

    if "INFO_MESSAGE" in results:
        message = results["INFO_MESSAGE"]
        # Use a helper function for text wrapping in PDF
        try:
            from reportlab.lib.enums import TA_LEFT
            from reportlab.platypus import Paragraph
            from reportlab.lib.styles import getSampleStyleSheet
            from reportlab.lib.units import inch

            styles = getSampleStyleSheet()
            style = styles['Normal']
            style.fontName = font_name
            style.fontSize = 10
            style.alignment = TA_LEFT

            p = Paragraph(message, style)
            # Calculate height needed for the wrapped paragraph
            w, h = p.wrapOn(c, width - 100, height) # Max width 500, max height will be calculated
            p.drawOn(c, 50, y - h) # Draw at 50, y - calculated height
            y -= h + 12 # Move y down by paragraph height + some padding

        except ImportError: # Fallback if platypus is not available or causes issues
            # Original simple wrapping
            wrapped_lines = wrap_text(message, font_name, 10, width - 100, c) # Adjust max_width
            for line in wrapped_lines:
                c.drawString(50, y, line)
                y -= 12
    else:
        c.drawString(50, y, "ID Variant")
        c.drawString(200, y, "Статус")
        c.line(50, y-5, 550, y-5)

        y -= 25
        c.setFont(font_name, 10)
        for rs, status in results.items(): # `results` is `clinvar_results`
            if y < 50:
                c.showPage()
                y = height - 50
            display_status = "Запись ClinVar не найдена" if status == "Not Found" else str(status)
            # Special handling for ClinVar results
            c.drawString(50, y, rs.upper())
            c.drawString(200, y, display_status)
            y -= 20

    c.save()
    print(f"📄 PDF Report saved: {filename}")
    files.download(filename)

# New helper function to process rs_numbers.txt
def process_rs_numbers_file(filename='rs_numbers.txt'):
    cleaned_rs_ids = set()
    file_found = True
    try:
        with open(filename, 'r') as f:
            raw_content = f.read()
            print(f"DEBUG: Raw content from '{filename}':\n---\n{raw_content}\n---")

            # Use regex to find patterns that look like 'rs' followed by numbers,
            # potentially with internal spaces. This will capture 'rs1000 000' as one unit.
            rs_pattern = r'rs\s*\d+(?:\s*\d+)*'
            found_rs_candidates = re.findall(rs_pattern, raw_content, re.IGNORECASE)
            print(f"DEBUG: Found RS candidates before processing: {found_rs_candidates}")

            for rs_candidate_raw in found_rs_candidates:
                # Remove ALL whitespace from the found candidate to get the final clean RS ID
                processed_rs_id = rs_candidate_raw.replace(" ", "")
                print(f"DEBUG: Processing '{rs_candidate_raw}' -> Cleaned: '{processed_rs_id}'")
                # After removing all spaces, ensure it still matches the base rsID format
                # (This re.match is now mostly a safety check, as the pattern should be robust)
                if re.match(r'^rs\d+$', processed_rs_id, re.IGNORECASE):
                    cleaned_rs_ids.add(processed_rs_id)
                else:
                    print(f"⚠️ Пропущен неверный формат RS ID: '{rs_candidate_raw}' в файле '{filename}' (после окончательной обработки: '{processed_rs_id}')")
        print(f"Загружено {len(cleaned_rs_ids)} уникальных RS-номеров из файла {filename}.")

    except FileNotFoundError:
        file_found = False
        print(f"Файл '{filename}' не найден.")
        return [], False # Return empty list and indicate file not found

    # Rewrite the file with one RS ID per line, sorted for consistency
    if cleaned_rs_ids:
        try:
            with open(filename, 'w') as f:
                for rs_id in sorted(list(cleaned_rs_ids)):
                    f.write(f"{rs_id}\n")
            print(f"✅ Файл '{filename}' обновлен с {len(cleaned_rs_ids)} уникальными RS-номерами (один на строку).")
        except Exception as e:
            print(f"❌ Ошибка при перезаписи файла '{filename}': {e}")

    return sorted(list(cleaned_rs_ids)), file_found

# --- Чтение ДНК последовательности из файла ---
try:
    with open('sample_dna.txt', 'r') as f:
        sample_dna = f.read().strip()
    print("✅ ДНК последовательность успешно загружена из 'sample_dna.txt'.")
except FileNotFoundError:
    sample_dna = ""
    print("❌ Файл 'sample_dna.txt' не найден. Используется пустая последовательность ДНК.")
except Exception as e:
    sample_dna = ""
    print(f"❌ Ошибка при чтении 'sample_dna.txt': {e}. Используется пустая последовательность ДНК.")

# --- Сравнение с эталонными изоформами ---
# Пример эталонных последовательностей экзонов (упрощенно)
exon_8_IIIb = "ATTATAGTAGAGAGA"
exon_9_IIIc = "ATTAGAGTAGAGACA"

# Для демонстрации возьмем начальный участок sample_dna, соответствующий длине экзона.
# В реальном сценарии может потребоваться более сложный поиск участка для сравнения.

exon_comparison_results = {}

if 'calculate_similarity' in globals(): # Ensure the function is defined
    segment_to_compare_8 = sample_dna[:len(exon_8_IIIb)]
    similarity_8 = calculate_similarity(segment_to_compare_8, exon_8_IIIb)
    exon_comparison_results["exon_8_IIIb"] = {"sequence": exon_8_IIIb, "similarity": similarity_8}

    segment_to_compare_9 = sample_dna[:len(exon_9_IIIc)]
    similarity_9 = calculate_similarity(segment_to_compare_9, exon_9_IIIc)
    exon_comparison_results["exon_9_IIIc"] = {"sequence": exon_9_IIIc, "similarity": similarity_9}
else:
    print("❌ Функция 'calculate_similarity' не определена. Сравнение экзонов пропущено.")


# Выполнение геномного сканирования
scanner_results = universal_genomic_scanner(sample_dna)

# Загрузка и обработка RS-номеров из файла rs_numbers.txt
rs_variants, rs_file_found = process_rs_numbers_file('rs_numbers.txt')

# Check if rs_variants is empty and set an info message if it is
if not rs_variants:
    # Use the more detailed message only if file was not found OR if it was found but empty
    if not rs_file_found or (rs_file_found and not rs_variants):
        print("Список RS-номеров пуст. В отчете ClinVar будет соответствующее сообщение.")
        clinvar_results = {"INFO_MESSAGE": "Раздел 'Клиническая Значимость (ClinVar)' не может быть сгенерирован. Файл 'rs_numbers.txt' не найден или оказался пустым. Для получения результатов, пожалуйста, создайте файл 'rs_numbers.txt' в корневой директории с одним rsID гена на каждой новой строке."}
    else: # Fallback for safety, though the above `if not rs_variants` covers it.
        clinvar_results = {"INFO_MESSAGE": "Не удалось найти валидные RS-номера для анализа в файле 'rs_numbers.txt'."}
else:
    clinvar_results = get_clinvar_significance_for_list(rs_variants)

# Генерация PDF отчета
save_report_to_pdf(clinvar_results, sample_dna, scanner_results, exon_comparison_results, "Genomic_Report.pdf")

✅ ДНК последовательность успешно загружена из 'sample_dna.txt'.

🚀 STARTING GENOMIC SCAN v4.0 (Visual)
📊 Global GC-Content: 52.56%
🧬 HTT Gene: 34 CAG repeats (NORMAL)

DEBUG: Raw content from 'rs_numbers.txt':
---
rs1000000
rs1137282
rs121908675
rs200
rs300
rs7069102

---
DEBUG: Found RS candidates before processing: ['rs1000000', 'rs1137282', 'rs121908675', 'rs200', 'rs300', 'rs7069102']
DEBUG: Processing 'rs1000000' -> Cleaned: 'rs1000000'
DEBUG: Processing 'rs1137282' -> Cleaned: 'rs1137282'
DEBUG: Processing 'rs121908675' -> Cleaned: 'rs121908675'
DEBUG: Processing 'rs200' -> Cleaned: 'rs200'
DEBUG: Processing 'rs300' -> Cleaned: 'rs300'
DEBUG: Processing 'rs7069102' -> Cleaned: 'rs7069102'
Загружено 6 уникальных RS-номеров из файла rs_numbers.txt.
✅ Файл 'rs_numbers.txt' обновлен с 6 уникальными RS-номерами (один на строку).
🔍 Запрос данных для 6 вариантов...


Поиск в базе:   0%|          | 0/6 [00:00<?, ?it/s]

ℹ️ Existing file 'Genomic_Report.pdf' removed.
📄 PDF Report saved: Genomic_Report.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Функция для сравнения последовательностей ДНК

Следующая функция `calculate_similarity` принимает две последовательности ДНК и возвращает процент сходства между ними. Она сравнивает нуклеотиды до минимальной длины обеих последовательностей и рассчитывает процент совпадений.

In [42]:
def calculate_similarity(seq1, seq2):
    # Приводим к одной длине для сравнения
    min_len = min(len(seq1), len(seq2))
    matches = sum(1 for a, b in zip(seq1[:min_len], seq2[:min_len]) if a == b)

    # Сходство рассчитывается на основе максимальной длины, чтобы учесть пропуски/вставки
    similarity = (matches / max(len(seq1), len(seq2))) * 100
    return round(similarity, 2)

print("✅ Функция calculate_similarity добавлена.")

✅ Функция calculate_similarity добавлена.


### Пример сравнения загруженной ДНК с эталонными изоформами

Теперь мы можем использовать эту функцию для сравнения `sample_dna` (которая загружается из `sample_dna.txt`) с заранее определенными эталонными последовательностями. В этом примере будут использоваться изоформы `exon_8_IIIb` и `exon_9_IIIc`.

In [43]:
# Пример эталонных последовательностей экзонов (упрощенно)
exon_8_IIIb = "ATTATAGTAGAGAGA"
exon_9_IIIc = "ATTAGAGTAGAGACA"

# Сравнение загруженной ДНК с эталонными изоформами
# Для демонстрации возьмем начальный участок sample_dna, соответствующий длине экзона.
# В реальном сценарии может потребоваться более сложный поиск участка для сравнения.

# Сравнение sample_dna с exon_8_IIIb
segment_to_compare_8 = sample_dna[:len(exon_8_IIIb)]
similarity_8 = calculate_similarity(segment_to_compare_8, exon_8_IIIb)
print(f"Сходство между загруженной ДНК и изоформой exon_8_IIIb: {similarity_8}%")

# Сравнение sample_dna с exon_9_IIIc
segment_to_compare_9 = sample_dna[:len(exon_9_IIIc)]
similarity_9 = calculate_similarity(segment_to_compare_9, exon_9_IIIc)
print(f"Сходство между загруженной ДНК и изоформой exon_9_IIIc: {similarity_9}%")

# Для справки, также сравним сами изоформы, как в вашем примере:
similarity_between_exons = calculate_similarity(exon_8_IIIb, exon_9_IIIc)
print(f"Сходство между изоформами IIIb и IIIc: {similarity_between_exons}%")

Сходство между загруженной ДНК и изоформой exon_8_IIIb: 40.0%
Сходство между загруженной ДНК и изоформой exon_9_IIIc: 53.33%
Сходство между изоформами IIIb и IIIc: 86.67%


In [5]:
%%writefile requirements.txt
biopython
reportlab
tqdm
matplotlib

Overwriting requirements.txt
